## State

In [173]:

from typing import TypedDict, Literal, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage


class RAGState(TypedDict):
    user_question: str
    messages: Annotated[list[BaseMessage], add_messages]
    grader_decision:Literal["relevant", "irrelevant"]
    context: list[str]
    router_decision: Literal["fetch", "use_context", "normal_chat"]
    old_rewritten_queries: list[str]
    current_rewritten_query: str
    retrievals: list[str]
    reranked_chunks: list[str]
    attempts_count: int
    web_search_result: list[str]

## LLM

In [174]:
from langchain_groq  import ChatGroq
import os 
from dotenv import load_dotenv

load_dotenv()
llm = ChatGroq(
      model="qwen/qwen3.8-27b",
      temperature=0.0,
      groq_api_key=os.getenv("groqAPIKey"),
      streaming=True,
      max_retries=0,
      )


## Router

In [186]:
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import END
from langchain_core.messages import SystemMessage, HumanMessage


router_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a query router for an Agentic RAG system.

        Classify the latest user question into exactly one category.

        1. use_context:
           Choose this ONLY when the existing conversation history
           or supplied context contains sufficient information to
           answer the latest question without retrieving more documents.

        2. fetch:
           Choose this when the question requires factual or technical
           information that is not sufficiently available in the
           existing conversation history or supplied context.
           This includes follow-up questions such as:
           "Why do we need it?", "How does it work?", and
           "What are its disadvantages?" when the available history
           does not contain the answer.

        3. normal_chat:
           Choose this for greetings, casual conversation, and
           questions that genuinely do not require knowledge retrieval.

        IMPORTANT:
        - Resolve pronouns and references using conversation history.
        - Do not classify a technical follow-up as normal_chat merely
          because it refers to a previous message.
        - A previous assistant answer is not automatically sufficient
          to answer every follow-up.
        - Prefer fetch when the available evidence is insufficient.
        - Return exactly one category:
          fetch, use_context, or normal_chat.
        - Do not answer the question.
        """
    ),
    (
        "human",
        """
        Conversation history:
        {conversation}

        Latest user question:
        {user_question}

        Existing context:
        {context}
        """
    )
])

def router_node(state: RAGState) -> dict:
    print("-------Router Node reacher-------")
    prompt_value = router_prompt.invoke({
        "conversation": state["messages"][:-1],
        "user_question": state["user_question"],
        "context": state["context"],
    })

    response = llm.invoke(prompt_value)

    decision = response.content.strip().lower()

    if decision not in {"fetch", "use_context", "normal_chat"}:
        raise ValueError(f"Invalid router decision: {decision}")
    print(f"----Router's decision :{decision}-----")

    return {"router_decision": decision}

## Query Rewritter

In [176]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate

rewriter_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a query rewriting component in an Agentic RAG system.

        Your task is to rewrite the user's question into a single,
        clear, self-contained search query that improves document retrieval.

        Instructions:
        1. Preserve the user's original intent.
        2. Use conversation history to resolve ambiguous references
           and understand the user's context.
        3. If previous rewritten queries are provided, learn from them
           and try a meaningfully different search formulation.
        4. If the previous query may have been too broad, narrow it.
           If it was too specific, broaden it appropriately.
        5. Include important technical terms, entities, and constraints.
        6. Do not answer the question.
        7. Do not invent details that are not supported by the question
           or conversation history.
        8. Return exactly one rewritten query as plain text.
        9. Do not include explanations, labels, numbering, or quotes.
        10.dont give empty response,try to give answer at worst if you cant cant rewrite give the same ,but try very hard
        """
    ),
    (
        "human",
        """
        Original user question:
        {user_question}

        Conversation history:
        {conversation}

        Previous rewritten queries:
        {rewritten_queries}

        Generate the next improved search query.
        """
    )
])

def rewriter(state: RAGState) -> dict:
    print(
    f"---------- reached rewriter node, "
    f"attempt count: {state['attempts_count'] + 1}"
)
    prompt_value = rewriter_prompt.invoke({
        "user_question": state["user_question"],
        "conversation": state["messages"][:-1],
        "rewritten_queries": state["old_rewritten_queries"],
    })

    response = llm.invoke(prompt_value)
    new_query = response.content.strip()
    
    print(f"-----rewritten query:{new_query}")

    if not new_query:
        raise ValueError("Rewriter returned an empty query")

    return {
        "old_rewritten_queries": [
            *state["old_rewritten_queries"],
            new_query
        ],
        "current_rewritten_query": new_query
}

## retriver

In [177]:
from time import perf_counter
import ollama


import os
from pinecone import Pinecone

pc = Pinecone(
    api_key=os.getenv("PINECONE_APIKEY")
)

index = pc.Index("ddia-ollama-vectors")
def pinecone_retriever(state: RAGState) -> dict:
    print("-----retriver node reached---------")
    start = perf_counter()

    query = state["current_rewritten_query"]

    t1 = perf_counter()
    result = ollama.embed(
        model="mxbai-embed-large",
        input=query
    )
    embedding_time = perf_counter() - t1

    query_vector = result["embeddings"][0]

    t2 = perf_counter()
    response = index.query(
        vector=query_vector,
        top_k=10,
        namespace="ddia",
        include_metadata=True
    )
    pinecone_time = perf_counter() - t2

    chunks = [
        match.metadata["text"]
        for match in response.matches
        if match.metadata and match.metadata.get("text")
    ]

    
    return {
        "retrievals": chunks,
        "attempts_count": state["attempts_count"] + 1,
    }

## reranker

In [178]:
from sentence_transformers import CrossEncoder

reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
from time import perf_counter


def reranker_node(state: RAGState) -> dict:
    print("---------- reached reranker ------------")
    query = state["current_rewritten_query"]
    chunks = state["retrievals"]
    
    start = perf_counter()
    if not chunks:
        return {"reranked_chunks": []}

    # Pair the query with every retrieved chunk
    pairs = [[query, chunk] for chunk in chunks]

    # Score each query-document pair
    scores = reranker_model.predict(pairs)
    print(f"CrossEncoder inference: {perf_counter() - start:.2f}s")



    # Sort chunks from most to least relevant
    ranked_chunks = sorted(
        zip(chunks, scores),
        key=lambda item: float(item[1]),
        reverse=True
    )

    # Keep the top 3 chunks
    reranked_chunks = [
        chunk for chunk, score in ranked_chunks[:3]
    ]

    return {"reranked_chunks": reranked_chunks}

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1579.20it/s]


## Grader

In [179]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate


class GradeOutput(BaseModel):
    decision: Literal["relevant", "irrelevant"] = Field(
        description="Whether the retrieved context is relevant to answering the question"
    )


grader_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a retrieval relevance grader.

        Given a user question and retrieved document chunks,
        decide whether the chunks contain information relevant
        to answering the question.

        Return "relevant" if at least one chunk contains useful
        information related to the question.

        Return "irrelevant" if none of the chunks are useful.

        Do not judge whether the final answer has been generated.
        Evaluate relevance only.
        """
    ),
    (
        "human",
        """
        User question:
        {user_question}

        Retrieved chunks:
        {reranked_chunks}
        """
    ),
])

grader_llm = llm.with_structured_output(GradeOutput)

grader_chain = grader_prompt | grader_llm


def grader_node(state: RAGState):
    print("----------- reached grader node--------")
    result = grader_chain.invoke({
        "user_question": state["user_question"],
        "reranked_chunks": "\n\n".join(
            state["reranked_chunks"]
        ),
    })

    return {
        "grader_decision": result.decision 
    }

## web search


In [180]:
import os
from tavily import TavilyClient

tavily_client = TavilyClient(
    api_key=os.getenv("TAVILY_API_KEY")
)


def web_search_node(state: RAGState) -> dict:
    print("---------reached web serach node ---------------")
    query = (
        state.get("current_rewritten_query")
        or state["user_question"]
    )

    response = tavily_client.search(query)

    lines = []

    for i, item in enumerate(response.get("results", [])):
        lines.append(
            f"{i + 1}. Title: {item.get('title', '')}\n"
            f"URL: {item.get('url', '')}\n"
            f"Content: {item.get('content', '')}"
        )

    return {"web_search_result": lines}

## final LLM node


In [181]:

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


final_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a helpful AI assistant.

        Answer the user's latest question using the available
        information and conversation history.

        Rules:
        - Use supplied context and search results when relevant.
        - Use history to resolve references such as "it", "that",
          and follow-up questions.
        - Do not treat old conversation as verified external evidence.
        - If information is insufficient, say so.
        - Do not invent facts.
        - Include source URLs when available.
        - Never return an empty response.
        """
    ),
    MessagesPlaceholder(variable_name="messages"),
    (
        "human",
        """
        Latest user question:
        {user_question}

        Available information:
        {context}
        """
    ),
])


def final_llm_node(state: RAGState) -> dict:
    print("---------- reached final LLM node ----------")

    available_information = (
        state.get("context", [])
        + state.get("reranked_chunks", [])
        + state.get("web_search_result", [])
    )

    prompt_value = final_prompt.invoke({
        # Exclude the latest HumanMessage because the latest
        # question is already supplied separately below.
        "messages": state["messages"][:-1],
        "user_question": state["user_question"],
        "context": "\n\n".join(available_information),
    })

    response = llm.invoke(prompt_value)

    answer = (
        response.content.strip()
        if isinstance(response.content, str)
        else ""
    )

    if not answer:
        answer = (
            "I couldn't generate a reliable answer. "
            "Please try rephrasing your question."
        )

    from langchain_core.messages import AIMessage

    return {
        "messages": [AIMessage(content=answer)]
    }


## Graph

In [182]:

from langgraph.graph import StateGraph, START, END


def route_after_router(state: RAGState) -> str:
    return state["router_decision"]


def route_after_grader(state: RAGState) -> str:
    if state["grader_decision"] == "relevant":
        return "final_llm"

    if state["attempts_count"] < 3:
        return "rewriter"

    return "web_search"


builder = StateGraph(RAGState)

builder.add_node("router", router_node)
builder.add_node("rewriter", rewriter)
builder.add_node("retriever", pinecone_retriever)
builder.add_node("reranker", reranker_node)
builder.add_node("grader", grader_node)
builder.add_node("web_search", web_search_node)
builder.add_node("final_llm", final_llm_node)


# Router is now the entry node
builder.add_edge(START, "router")

builder.add_conditional_edges(
    "router",
    route_after_router,
    {
        "fetch": "rewriter",
        "use_context": "final_llm",
        "normal_chat": "final_llm",
    },
)

builder.add_edge("rewriter", "retriever")
builder.add_edge("retriever", "reranker")
builder.add_edge("reranker", "grader")

builder.add_conditional_edges(
    "grader",
    route_after_grader,
    {
        "final_llm": "final_llm",
        "rewriter": "rewriter",
        "web_search": "web_search",
    },
)

builder.add_edge("web_search", "final_llm")
builder.add_edge("final_llm", END)

graph = builder.compile()


In [189]:

from langchain_core.messages import HumanMessage, AIMessage

# Persistent history for this CLI conversation.
conversation_history = []


while True:
    query = input("You: ").strip()

    if query.lower() == "exit":
        break

    if not query:
        continue

    # Add the latest user message before running the graph.
    current_messages = [
        *conversation_history,
        HumanMessage(content=query),
    ]

    # Create fresh query-specific state for every turn.
    current_state: RAGState = {
        "user_question": query,
        "messages": current_messages,
        "grader_decision": "irrelevant",
        "context": [],
        "router_decision": "fetch",
        "old_rewritten_queries": [],
        "current_rewritten_query": "",
        "retrievals": [],
        "reranked_chunks": [],
        "attempts_count": 0,
        "web_search_result": [],
    }

    # Fully synchronous graph execution.
    result = graph.invoke(current_state)

    # The graph returns the updated message history.
    conversation_history = result["messages"]

    # The final message should be the assistant's response.
    answer_message = conversation_history[-1]

    print("\nAssistant:")
    print(answer_message.content)
    print()


-------Router Node reacher-------
----Router's decision :fetch-----
---------- reached rewriter node, attempt count: 1
-----rewritten query:purpose and benefits of message queues in software architecture
-----retriver node reached---------
---------- reached reranker ------------
CrossEncoder inference: 0.52s
----------- reached grader node--------
---------- reached final LLM node ----------

Assistant:
Based on the provided information, message queues (or message brokers) are needed because they offer several key advantages over direct communication methods like Remote Procedure Calls (RPC):

1.  **Improved Reliability and Buffering:** They act as a buffer when the recipient is unavailable or overloaded, which enhances system reliability.
2.  **Prevention of Message Loss:** They can automatically redeliver messages to processes that have crashed, ensuring that messages are not lost.
3.  **Decoupling of Sender and Recipient:** They logically decouple the sender from the recipient. The

In [185]:
initial_state["messages"]

[]